# LightFM Model Notebook

`LIGHTFM_NOTEBOOK_EXECUTION_PLAN.md`의 실행 책임 단위를 그대로 따른다.
각 Unit은 Markdown 주석 셀 + 코드 셀 한 쌍으로 관리한다.

In [1]:
import sys
print(sys.executable)
from lightfm import LightFM
print("lightfm ok")

c:\dev\project\SKN27-FINAL-1Team\ai\experiments\.venv\Scripts\python.exe
lightfm ok


c:\dev\project\SKN27-FINAL-1Team\ai\experiments\.venv\Lib\site-packages\lightfm\_lightfm_fast.py:9: UserWarning: LightFM was compiled without OpenMP support. Only a single thread will be used.
  warnings.warn(


### Unit 1 - 환경 및 공통 설정
- Why: 노트북 실행 재현성을 확보한다.
- Input: Python 환경, 라이브러리(lightfm/pandas/numpy)
- Output: import 완료, 경로/상수 정의
- Check: import 에러 없음
- DoD: 공통 상수와 seed가 선언되어 다음 Unit에서 재사용 가능

In [2]:
# Unit 1
from pathlib import Path
import random

import numpy as np
import pandas as pd

from lightfm import LightFM
from lightfm.data import Dataset
from lightfm.cross_validation import random_train_test_split
from lightfm.evaluation import precision_at_k, recall_at_k

SEED = 42
TEST_RATIO = 0.2
EPOCHS = 30
NUM_THREADS = 2

random.seed(SEED)
np.random.seed(SEED)

DATA_PATH = Path("review_by_llm.csv")
print({"seed": SEED, "test_ratio": TEST_RATIO, "epochs": EPOCHS})

{'seed': 42, 'test_ratio': 0.2, 'epochs': 30}


### Unit 2 - 데이터 로드 및 필수 컬럼 검증
- Why: 학습 입력 정합성을 먼저 보장해야 한다.
- Input: `review_by_llm.csv`
- Output: 검증 완료 DataFrame
- Check: 필수 컬럼 누락/결측/타입 이상 확인
- DoD: 필수 컬럼이 모두 존재하고 기본 통계가 출력됨

In [ ]:
# Unit 2
required_cols = ["recipe_id", "group_id", "star_count", "star_norm", "positive", "negative"]

df = pd.read_csv(DATA_PATH)
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

print("rows:", len(df))
print("null_counts:\n", df[required_cols].isna().sum())
print(df[required_cols].head(3))

### Unit 3 - interaction 타겟 생성
- Why: 실험별 학습 신호를 일관된 방식으로 생성한다.
- Input: star/sentiment 컬럼
- Output: `interaction_value`
- Check: 타겟 분포와 min/max 확인
- DoD: 선택한 타겟 방식이 명시되고 컬럼 생성 완료

In [ ]:
# Unit 3
TARGET_MODE = "binary"  # binary | star_norm | sentiment | hybrid

if TARGET_MODE == "binary":
    df["interaction_value"] = 1.0
elif TARGET_MODE == "star_norm":
    df["interaction_value"] = df["star_norm"].astype(float)
elif TARGET_MODE == "sentiment":
    df["interaction_value"] = (df["positive"] - df["negative"]).astype(float)
elif TARGET_MODE == "hybrid":
    df["interaction_value"] = (df["star_norm"] + (df["positive"] - df["negative"]))
else:
    raise ValueError(f"Unsupported TARGET_MODE: {TARGET_MODE}")

print("target_mode:", TARGET_MODE)
print(df["interaction_value"].describe())

### Unit 4 - ID 매핑 및 Dataset 구성
- Why: user/item 내부 인덱스 매핑을 고정한다.
- Input: `group_id`, `recipe_id`
- Output: LightFM Dataset
- Check: user/item 개수 출력
- DoD: `dataset.fit()` 완료

In [ ]:
# Unit 4
user_ids = df["group_id"].astype(str).unique().tolist()
item_ids = df["recipe_id"].astype(str).unique().tolist()

dataset = Dataset()
dataset.fit(users=user_ids, items=item_ids)

print({"users": len(user_ids), "items": len(item_ids)})

### Unit 5 - interactions matrix 생성
- Why: 학습/평가 입력 sparse matrix를 만든다.
- Input: `(user, item, interaction_value)`
- Output: interactions matrix
- Check: matrix shape, nnz 출력
- DoD: interactions 생성 및 sparsity 계산 가능

In [ ]:
# Unit 5
triples = list(
    zip(
        df["group_id"].astype(str),
        df["recipe_id"].astype(str),
        df["interaction_value"].astype(float),
    )
)

interactions, _ = dataset.build_interactions(triples)
num_users, num_items = interactions.shape
density = interactions.nnz / (num_users * num_items) if num_users and num_items else 0.0

print({"shape": interactions.shape, "nnz": interactions.nnz, "density": density})

### Unit 6 - train/test 분할
- Why: 오프라인 평가를 위한 검증 셋을 분리한다.
- Input: interactions
- Output: train, test
- Check: train/test nnz 출력
- DoD: 재현 가능한 분할(random_state 고정) 완료

In [ ]:
# Unit 6
train, test = random_train_test_split(
    interactions,
    test_percentage=TEST_RATIO,
    random_state=np.random.RandomState(SEED),
)

print({"train_nnz": train.nnz, "test_nnz": test.nnz})

### Unit 7 - LightFM 학습
- Why: 랭킹 모델을 학습해 추천 점수를 생성한다.
- Input: train matrix
- Output: 학습된 model
- Check: 학습 완료, train Precision@5 추이 확인
- DoD: 지정 epoch 학습이 종료되고 최소 지표가 산출됨

In [ ]:
# Unit 7
model = LightFM(loss="warp", random_state=SEED)
train_p5_history = []

for _ in range(EPOCHS):
    model.fit_partial(train, epochs=1, num_threads=NUM_THREADS)
    p5 = precision_at_k(model, train, k=5).mean()
    train_p5_history.append(float(p5))

print({"last_train_precision@5": train_p5_history[-1] if train_p5_history else None})

### Unit 8 - Precision/Recall 평가
- Why: 모델의 추천 품질을 수치화한다.
- Input: model, test matrix
- Output: Precision@K, Recall@K
- Check: K=5/10 지표 출력
- DoD: 핵심 지표 4개(p@5, p@10, r@5, r@10) 기록

In [ ]:
# Unit 8
metrics = {
    "precision@5": float(precision_at_k(model, test, k=5).mean()),
    "precision@10": float(precision_at_k(model, test, k=10).mean()),
    "recall@5": float(recall_at_k(model, test, k=5).mean()),
    "recall@10": float(recall_at_k(model, test, k=10).mean()),
}
print(metrics)

### Unit 9 - baseline 비교
- Why: LightFM 성능 우위를 판단한다.
- Input: LightFM 지표, baseline 지표
- Output: 개선/동등/열위 판정
- Check: 비교표 및 판정 로그
- DoD: Go/No-Go 초안 기준으로 결과 상태가 결정됨

In [ ]:
# Unit 9
# TODO: baseline 계산 로직(인기 기반)을 연결해 아래 dict를 실제 값으로 채운다.
baseline_metrics = {
    "precision@10": None,
    "recall@10": None,
}

go_no_go = "pending"
if baseline_metrics["precision@10"] is not None and baseline_metrics["recall@10"] is not None:
    if (
        metrics["precision@10"] >= baseline_metrics["precision@10"]
        and metrics["recall@10"] >= baseline_metrics["recall@10"]
    ):
        go_no_go = "go"
    else:
        go_no_go = "no_go"

print({"baseline": baseline_metrics, "decision": go_no_go})

### Unit 10 - 실험 리포트 기록
- Why: 실험 설정/결과/판단을 문서 기준으로 누적한다.
- Input: 실행 설정, metrics, baseline 비교 결과
- Output: 구조화된 실험 요약
- Check: 필수 필드 누락 여부
- DoD: 결과가 재현 가능한 형식으로 남음

In [ ]:
# Unit 10
experiment_report = {
    "data_path": str(DATA_PATH),
    "target_mode": TARGET_MODE,
    "seed": SEED,
    "test_ratio": TEST_RATIO,
    "epochs": EPOCHS,
    "loss": "warp",
    "matrix": {
        "num_users": int(interactions.shape[0]),
        "num_items": int(interactions.shape[1]),
        "nnz": int(interactions.nnz),
    },
    "metrics": metrics,
    "decision": go_no_go,
}

print(experiment_report)